# What's New in gVXR 2.1.1

This tutorial introduces the major new features added in **gVXR 2.1.1**. It is intended for users who are already familiar with the basics of gVXR, including loading geometries, defining X-ray sources and detectors, and computing radiographs.

## Learning objectives

By the end of this tutorial, you will be able to:

1. Navigate and inspect a scene using the new camera system.
2. Interactively adjust acquisition geometry using the mouse and keyboard.
3. Load scanner parameters directly from an XTekCT configuration file.
4. Check whether points lie within an object's bounding box.
5. Compute and export object silhouettes.
6. Use `gVXRDataReader` with both file-based and in-memory data.
7. Understand recent API changes and deprecated functionality.
8. Take advantage of improved visualisation and interaction features.

---

## Table of Contents

1. [Navigating a Scene with the New Camera System](#1-navigating-a-scene-with-the-new-camera-system)
2. [Rotating the Sample Interactively](#2-rotating-the-sample-interactively)
3. [Interactively Adjusting Imaging Geometry](#3-interactively-adjusting-imaging-geometry)
4. [Panning the View](#4-panning-the-view)
5. [Built-in Help](#5-built-in-help)
6. [Copying Screenshots to the Clipboard](#6-copying-screenshots-to-the-clipboard)
7. [Importing XTekCT Geometries](#7-importing-xtekct-geometries)
8. [Bounding-Box Queries](#8-bounding-box-queries)
9. [Computing Silhouettes](#9-computing-silhouettes)
10. [Reading Data from Memory](#10-reading-data-from-memory)
11. [Projection Matrices with Explicit Units](#11-projection-matrices-with-explicit-units)
12. [Visualisation Improvements](#12-visualisation-improvements)
13. [Migration Notes](#migration-notes)
14. [Capstone Exercise](#capstone-exercise)

---

# 1. Navigating a Scene with the New Camera System

## Overview

gVXR now provides a dedicated camera system that supports:

- Rotating around the scene;
- Panning the view;
- Zooming in and out;
- Direct manipulation of camera parameters.

This behaviour is similar to modern CAD and scientific visualisation software and makes scene exploration considerably more intuitive.

## Exercise

### Objective

Explore the new camera system and become familiar with scene navigation.

### Steps

1. Load a sample.
2. Start the interactive viewer:

   ```python
   gvxr.renderLoop()
   ```

or
3. Use the `ViewerPanel` Jupyter Widget.

```python
from gvxrPython3.notebook import ViewerPanel
...
ViewerPanel()
```

4. Rotate around the sample.
5. Zoom in on a small feature.
6. Pan across the scene.
7. Inspect the sample from different viewpoints.

### Expected output

You should be able to:

- Rotate the camera around the sample;
- Zoom smoothly towards and away from the sample;
- Pan the view without moving the sample;
- Inspect the sample from arbitrary viewpoints.

### Questions

- What is the difference between moving the camera and rotating the sample?
- Which approach is preferable when inspecting a complex assembly?
- How does camera navigation simplify scene exploration?

## Reference Implementation

In [ ]:
import numpy as np
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        40,
        40,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

# Rotate the sample. The spheres are within the cylinder, only rotate the later.
gvxr.rotateNode("cylinder", 90, 1, 0, 0);

gvxr.applySOD("cylinder", gvxr.getBestSOD("cylinder", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

gvxr.computeXRayImage();
gvxr.displayScene()

gvxr.renderLoop()

In [ ]:
from gvxrPython3.notebook import ViewerPanel
ViewerPanel()

In [ ]:
gvxr.destroy()

---

# 2. Rotating the Sample Interactively

## Overview

The sample can now be rotated directly using the right mouse button and mouse movement.

This allows users to quickly investigate how object orientation affects projections.

## Exercise

### Objective

Investigate the effect of sample orientation on generated projections.

### Steps

1. Load a mesh:

   ```python
   gvxr.loadMeshFile(...)
   ```

2. Open the interactive viewer.
3. Rotate the sample using the right mouse button.
4. Observe how the projected shape changes.
5. Compare several viewing angles.

### Expected output

You should observe:

- A smooth rotation of the object in the viewer;
- Changes in the object's orientation relative to the beam;
- Corresponding changes in the projected silhouette and radiograph.

### Questions

- Which orientations produce the largest projected area?
- Which features become hidden in certain views?
- How would object orientation affect a tomography acquisition?

### Challenge (optional)

Try the same experiment with a CAD assembly containing multiple components and identify orientations that produce overlapping structures.

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        40,
        40,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

# Rotate the sample. The spheres are within the cylinder, only rotate the later.
gvxr.rotateNode("cylinder", 90, 1, 0, 0);

gvxr.applySOD("cylinder", gvxr.getBestSOD("cylinder", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

gvxr.computeXRayImage();
gvxr.displayScene()

print("Use the right mouse button to rotate the sample.")

gvxr.renderLoop()
gvxr.destroy()

---

# 3. Interactively Adjusting Imaging Geometry

## Overview

The source-to-object distance (SOD) and object-to-detector distance (ODD) can now be adjusted interactively using the keyboard and mouse.

Collision detection prevents invalid configurations.

## Exercise

### Objective

Investigate the effect of geometry changes on magnification.

### Steps

1. Load a sample.
2. Increase and decrease the SOD using:

   ```text
   S + ↑
   S + ↓
   ```

3. Increase and decrease the ODD using:

   ```text
   D + ↑
   D + ↓
   ```

4. Repeat using the mouse wheel.
5. Repeat using mouse dragging.
6. Observe the projection size on the detector.

### Expected output

You should observe:

- An increase in magnification as the source approaches the sample;
- Changes in projected object size as the detector position changes;
- Automatic prevention of source or detector collisions with the sample.

### Questions

- How does magnification change as the source approaches the sample?
- How does detector position affect the field of view?
- When does collision detection prevent further movement?

### Challenge (optional)

Determine a geometry that maximises magnification without clipping the object from the detector.

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        40,
        40,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

# Rotate the sample. The spheres are within the cylinder, only rotate the later.
gvxr.rotateNode("cylinder", 90, 1, 0, 0);

gvxr.applySOD("cylinder", gvxr.getBestSOD("cylinder", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

gvxr.computeXRayImage();
gvxr.displayScene()

print("Adjust geometry interactively:")
print("  S + arrows / mouse = source")
print("  D + arrows / mouse = detector")

gvxr.renderLoop()
gvxr.destroy()

---

# 4. Panning the View

## Overview

The camera can now be translated laterally, making it easier to inspect large scenes and off-centre objects.

## Exercise

### Objective

Use camera panning to explore a scene.

### Steps

1. Load a scene containing several objects.
2. Zoom in on one object using the mouse wheel.
3. Pan the camera across the scene using the middle mouse button.
4. Inspect every object without moving the sample itself.

### Expected output

You should be able to:

- Move the viewpoint horizontally and vertically;
- Inspect different areas of the scene;
- Keep the sample position unchanged while exploring the environment.

### Questions

- When is panning preferable to zooming?
- Does panning help inspect large assemblies?
- How does panning improve scene navigation?

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
sphere_radius_in_mm = 2.0

gvxr.makeCylinder(
        "cylinder",
        150,
        cylinder_height_in_mm,
        cylinder_radius_in_mm,
        "mm")

gvxr.makeSphere("sphere",
        50,
        50,
        sphere_radius_in_mm,
        "mm")

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("sphere", 0, 1, 0, 1)

# translate parts of the sample.
gvxr.translateNode("cylinder", 0, 20, 0, "mm")
gvxr.translateNode("sphere", 100, 0, 20, "mm")

gvxr.applySOD("root", gvxr.getBestSOD("root", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("sphere", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("sphere");

gvxr.computeXRayImage();
gvxr.displayScene()

print("Adjust view:")
print("  Use the mouse wheel to zoom in and out.")
print("  Use the middle mouse button to pan across the scene.")

gvxr.renderLoop()
gvxr.destroy()

---

# 5. Built-in Help

## Overview

A help screen is now available directly from the interactive viewer.

## Exercise

### Objective

Become familiar with all available keyboard and mouse controls.

### Steps

1. Open the interactive viewer.
2. Display the help screen.
3. Review all available controls.
4. Experiment with the navigation shortcuts.
5. Experiment with the geometry-manipulation shortcuts.

### Expected output

You should see:

- A list of keyboard shortcuts;
- A list of mouse controls;
- Descriptions of available interactions.

### Questions

- Which controls are most useful for your workflow?
- Which controls were not available in previous releases?

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        40,
        40,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

# Rotate the sample. The spheres are within the cylinder, only rotate the later.
gvxr.rotateNode("cylinder", 90, 1, 0, 0);

gvxr.applySOD("cylinder", gvxr.getBestSOD("cylinder", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

gvxr.computeXRayImage();
gvxr.displayScene()

print("Press H to hide/show the help message.")

gvxr.renderLoop()
gvxr.destroy()

---

# 6. Copying Screenshots to the Clipboard

## Overview

Two kinds of image can now be copied directly to the system clipboard from the interactive viewer, without saving a file first:

| Shortcut | What is copied |
|----------|----------------|
| **Ctrl+C** | A screenshot of the 3D rendering (the scene as displayed in the viewer window) |
| **Ctrl+Shift+C** | The X-ray image (the simulated projection, i.e. the detector output) |

This simplifies creating presentations, documentation, tutorials, and bug reports.

## Exercise

### Objective

Capture and reuse both the 3D rendering and the X-ray image produced by gVXR.

### Steps

1. Run the reference implementation below to open the interactive viewer.
2. Rotate/zoom the scene until the sample is nicely framed.
3. Press **Ctrl+C** to copy a screenshot of the 3D rendering.
4. Paste it into a presentation, document, or e-mail (Ctrl+V).
5. Press **Ctrl+Shift+C** to copy the X-ray image.
6. Paste it in the same document, next to the first image.

### Expected output

You should obtain:

- Two images pasted side by side: the 3D view of the scene and the corresponding X-ray projection;
- The 3D screenshot shows the scene as displayed in the viewer (including the camera view you chose);
- The X-ray image is a faithful reproduction of the projection computed by gVXR.

### Questions

- Does each image appear identical after pasting?
- What is the difference between the two images in terms of content and resolution? (Hint: compare with the detector size, 640 x 320 pixels.)
- How could these features simplify bug reporting or teaching activities?

### Challenge (optional)

Create a short slide deck illustrating different acquisition geometries: for each geometry, paste the 3D rendering (Ctrl+C) and the resulting X-ray image (Ctrl+Shift+C).

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        40,
        40,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

# Rotate the sample. The spheres are within the cylinder, only rotate the later.
gvxr.rotateNode("cylinder", 90, 1, 0, 0);

gvxr.applySOD("cylinder", gvxr.getBestSOD("cylinder", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

gvxr.computeXRayImage();
gvxr.displayScene()

print("Press Ctrl+C to copy a screenshot of the 3D rendering to the clipboard.")
print("  Paste it (Ctrl+V) into a slideshow for a presentation.")
print("Press Ctrl+Shift+C to copy the X-ray image to the clipboard.")
print("  Paste it (Ctrl+V) into a slideshow for a presentation.")

gvxr.renderLoop()
gvxr.destroy()

---

# 7. Importing XTekCT Geometries

## Overview

Acquisition parameters can now be imported directly from an `.xtekct` file.

## Exercise

### Objective

Configure an acquisition geometry from an XTekCT configuration file.

### Steps

1. Obtain an `.xtekct` file.
2. Import the acquisition parameters into gVXR.
3. Inspect the generated geometry.
4. Compare it with a manually configured setup.

### Expected output

You should obtain:

- A fully configured acquisition geometry;
- Correct source and detector positions;
- Matching geometry between the imported and manually configured setups.

### Questions

- Which parameters are imported automatically?
- Are the imported and manual geometries identical?
- How much setup time can be saved?

### Challenge (optional)

Import several XTekCT files from different scanners and compare their geometries.

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

# Load the XTekCT file
gvxr.loadXtekct("../data/test.xtekct")



# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        50,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        20,
        20,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

print("SOD", gvxr.getSOD("root", "mm"), "mm")
# SrcToObject=152.818

print("SDD", gvxr.getSDD("mm"), "mm")
# SrcToDetector=1404.023

print("Number of detector pixels:", gvxr.getDetectorNumberOfPixels())
# DetectorPixelsX=512
# DetectorPixelsY=512

print("Detector pixel pitch:", gvxr.getDetectorPixelSpacing("mm"))
# DetectorPixelSizeX=0.8
# DetectorPixelSizeY=0.8

print("White level:", gvxr.getWhiteLevel())
# WhiteLevel=60000

# [Xrays]
print("Tube voltage:", gvxr.getVoltage("kV"), "kV")
# XraykV=125

# [CTPro]
print("Filter material:", gvxr.getFiltrationMaterial())
print("Filter thickness:", gvxr.getFiltrationThickness("mm"))
# Filter_ThicknessMM=0.500
# Filter_Material=Tin

gvxr.computeXRayImage();
gvxr.displayScene()

gvxr.renderLoop()
gvxr.destroy()

---

# 8. Bounding-Box Queries

## Overview

The function `isPointInNodeAndChildrenBoundingBox()` determines whether a point is inside the bounding box of an object or any of its descendants.

## Exercise

### Objective

Use bounding-box queries to validate points in a scene.

### Steps

1. Load a sample.
2. Define several test coordinates.
3. Evaluate each coordinate using:

   ```cpp
   isPointInNodeAndChildrenBoundingBox(...)
   ```

4. Record the results.

### Expected output

You should obtain:

- A set of coordinates classified as inside or outside the bounding box;
- Consistent results for all tested points.

### Questions

- Which points are inside the bounding box?
- Which points are outside?
- How could this functionality support collision detection?

<!--### Challenge (optional)

Implement a simple picking tool that reports whether the mouse cursor intersects a bounding box.-->

## Reference Implementation

In [ ]:
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.destroy()
gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
sphere_radius_in_mm = 2.0

gvxr.makeCylinder(
        "cylinder",
        150,
        cylinder_height_in_mm,
        cylinder_radius_in_mm,
        "mm")

gvxr.makeSphere("sphere",
        50,
        50,
        sphere_radius_in_mm,
        "mm")

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("sphere", 0, 1, 0, 1)

# translate parts of the sample.
cylinder_position = [0.0, 20.0, 0.0]
sphere_position = [80, 0, 20]

gvxr.translateNode("cylinder", *cylinder_position, "mm")
gvxr.translateNode("sphere", *sphere_position, "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("sphere", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("sphere");


root_centre = gvxr.getNodeAndChildrenBoundingBoxCenter("root", "mm")
cylinder_centre = gvxr.getNodeAndChildrenBoundingBoxCenter("cylinder", "mm")
sphere_centre = gvxr.getNodeAndChildrenBoundingBoxCenter("sphere", "mm")

print("Bounding box centre of whole sample:", root_centre, "mm")
print("Bounding box centre of cylinder:", cylinder_centre, "mm")
print("Bounding box centre of sphere:", sphere_centre, "mm")
print()

root_bbox = gvxr.getNodeAndChildrenBoundingBox("root", "mm")
cylinder_bbox = gvxr.getNodeAndChildrenBoundingBox("cylinder", "mm")
sphere_bbox = gvxr.getNodeAndChildrenBoundingBox("sphere", "mm")

print("Bounding box of whole sample:", root_bbox, "mm")
print("Bounding box of cylinder:", cylinder_bbox, "mm")
print("Bounding box of sphere:", sphere_bbox, "mm")
print()

# Origin
print("Test origin")
origin_position = [0.0, 0.0, 0.0]
inside_root = gvxr.isPointInNodeAndChildrenBoundingBox(*origin_position, "mm", "root")
inside_cylinder = gvxr.isPointInNodeAndChildrenBoundingBox(*origin_position, "mm", "cylinder")
inside_sphere = gvxr.isPointInNodeAndChildrenBoundingBox(*origin_position, "mm", "sphere")
print(f"{origin_position} inside bounding box ({root_bbox}): {inside_root}")
print(f"{origin_position} inside bounding box ({cylinder_bbox}): {inside_cylinder}")
print(f"{origin_position} inside bounding box ({sphere_bbox}): {inside_sphere}")
print()

# Cylinder centre
print("Test cylinder centre")
inside_root = gvxr.isPointInNodeAndChildrenBoundingBox(*cylinder_position, "mm", "root")
inside_cylinder = gvxr.isPointInNodeAndChildrenBoundingBox(*cylinder_position, "mm", "cylinder")
inside_sphere = gvxr.isPointInNodeAndChildrenBoundingBox(*cylinder_position, "mm", "sphere")
print(f"{cylinder_position} inside bounding box ({root_bbox}): {inside_root}")
print(f"{cylinder_position} inside bounding box ({cylinder_bbox}): {inside_cylinder}")
print(f"{cylinder_position} inside bounding box ({sphere_bbox}): {inside_sphere}")
print()

# Sphere centre
print("Test sphere centre")
inside_root = gvxr.isPointInNodeAndChildrenBoundingBox(*sphere_position, "mm", "root")
inside_cylinder = gvxr.isPointInNodeAndChildrenBoundingBox(*sphere_position, "mm", "cylinder")
inside_sphere = gvxr.isPointInNodeAndChildrenBoundingBox(*sphere_position, "mm", "sphere")
print(f"{sphere_position} inside bounding box ({root_bbox}): {inside_root}")
print(f"{sphere_position} inside bounding box ({cylinder_bbox}): {inside_cylinder}")
print(f"{sphere_position} inside bounding box ({sphere_bbox}): {inside_sphere}")
print()

gvxr.renderLoop()

gvxr.destroy()

---

# 9. Computing Silhouettes

## Overview

Silhouettes provide a binary representation of object occupancy and are significantly smaller than L-buffer images.

## Exercise

### Objective

Generate and save a silhouette image.

### Steps

1. Compute the silhouette:

   ```python
   gvxr.computeSilhouette("sample")
   ```

2. Retrieve it:

   ```python
   silhouette = gvxr.getLastSilhouette()
   ```

3. Save it:

   ```python
   gvxr.saveLastSilhouette("silhouette.tif")
   ```

4. Visualise the resulting image.

### Expected output

A binary image where:

- Background pixels have value 0;
- Object pixels have value 255;
- The object's projected shape is clearly visible;
- The silhouette file is saved successfully.

### Questions

- Which regions correspond to the object?
- Which regions correspond to the background?
- How does the silhouette differ from an X-ray projection?

### Challenge (optional)

Generate silhouettes from many viewing directions and build a dataset for machine-learning applications such as classification, detection, or pose estimation.

## Reference Implementation


In [ ]:
from sys import getsizeof
import numpy as np
import matplotlib.pyplot as plt # Plotting
from tifffile import imwrite
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0

gvxr.makeCylinder(
        "sample",
        150,
        cylinder_height_in_mm,
        cylinder_radius_in_mm,
        "mm")

# Change the colours
gvxr.setColour("sample", 1, 0, 0, 1)

# translate the sample.
cylinder_position = [0.0, 20.0, 0.0]

gvxr.translateNode("sample", *cylinder_position, "mm")

gvxr.setElement("sample", "Si");

gvxr.addPolygonMeshAsInnerSurface("sample");

# Compute the silhouette of the cylinder
cylinder_silhouette = gvxr.computeSilhouette("sample")

# Retrieve the last computed silhouette
cylinder_silhouette = gvxr.getLastSilhouette()

# Save them into TIFF files
imwrite("cylinder_silhouette1.tif", np.array(cylinder_silhouette, dtype="uint8"))
gvxr.saveLastSilhouette("cylinder_silhouette2.tif")

gvxr.renderLoop()

gvxr.destroy()

## Additional Exercise

### Objective

Compare silhouettes and L-buffer images.

### Steps

1. Compute an L-buffer:

   ```python
   gvxr.computeLBuffer()
   ```

2. Compute a silhouette:

   ```python
   gvxr.computeSilhouette()
   ```

3. Compare memory usage.
4. Compare visual appearance.
5. Compare suitability for segmentation tasks.

### Expected output

You should observe:

- A floating-point image for the L-buffer;
- A binary image for the silhouette;
- Smaller memory usage for the silhouette;
- Easier segmentation of the silhouette image.

### Questions

- Which representation is more compact?
- Which representation contains more physical information?
- Which representation is more suitable for machine-learning pipelines?

## Reference Implementation


In [ ]:
from sys import getsizeof
import numpy as np
import matplotlib.pyplot as plt # Plotting
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()
gvxr.hideWindow()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
sphere_radius_in_mm = 15.0

gvxr.makeCylinder(
        "cylinder",
        150,
        cylinder_height_in_mm,
        cylinder_radius_in_mm,
        "mm")

gvxr.makeSphere("sphere",
        50,
        50,
        sphere_radius_in_mm,
        "mm")

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("sphere", 0, 1, 0, 1)

# translate parts of the sample.
cylinder_position = [0.0, 20.0, 0.0]
sphere_position = [80, 0, 20]

gvxr.translateNode("cylinder", *cylinder_position, "mm")
gvxr.translateNode("sphere", *sphere_position, "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("sphere", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("sphere");

In [ ]:
# Compute the silhouette of the cylinder and convert it into a NumPy array
cylinder_silhouette = np.array(gvxr.computeSilhouette("cylinder"), dtype=np.uint8)

# Compute the silhouette of the sphere and convert it into a NumPy array
sphere_silhouette = np.array(gvxr.computeSilhouette("sphere"), dtype=np.uint8)

# Compute the path length of the X-rays through the cylinder and convert it into a NumPy array
cylinder_l_buffer = np.array(gvxr.computePathLength("cylinder"), dtype=np.single)

# Compute the path length of the X-rays through the cylinder and convert it into a NumPy array
sphere_l_buffer = np.array(gvxr.computePathLength("sphere"), dtype=np.single)

print("Size of silhouette:", getsizeof(cylinder_silhouette))
print("Size of path length:", getsizeof(cylinder_l_buffer))


In [ ]:
# Visualise the resulting image.
plt.figure(figsize=(10, 10))
plt.subplot(221)
plt.title("Cylinder path length")
plt.imshow(cylinder_l_buffer, cmap="gray")
plt.colorbar()
plt.subplot(222)
plt.title("Sphere path length")
plt.imshow(sphere_l_buffer, cmap="gray")
plt.colorbar()

plt.subplot(223)
plt.title("Cylinder silhouette")
plt.imshow(cylinder_silhouette, cmap="gray")
plt.colorbar()

plt.subplot(224)
plt.title("Sphere silhouette")
plt.imshow(sphere_silhouette, cmap="gray")
plt.colorbar()

plt.show()

In [ ]:
# Convert binary images (0 or 255) to boolean masks
mask1 = cylinder_silhouette == 255
mask2 = sphere_silhouette == 255

# Combine using bit flags
combined = mask1.astype(np.uint8) | (mask2.astype(np.uint8) << 1)

# Visualise the resulting image.
plt.figure()
plt.imshow(combined)
plt.colorbar()
plt.show()

In [ ]:
gvxr.destroy()

---

# 10. Reading Data from Memory

## Overview

`gVXRDataReader` can now work with data stored directly in memory.

## Exercise

### Objective

Compare disk-based and memory-based workflows.

### Steps

1. Create a volume in memory.
2. Load it using `gVXRDataReader`.
3. Load the same volume from disk.
4. Measure execution times.
5. Compare both workflows.

### Expected output

You should obtain:

- Successful loading from memory;
- Successful loading from disk;
- Timing measurements for both approaches.

### Questions

- Which workflow is faster?
- Is disk I/O a bottleneck?
- Which applications benefit most from in-memory data?

### Challenge (optional)

Create a workflow in which a reconstruction is generated and processed entirely in memory without writing intermediate files.

## Reference Implementation

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
from gvxrPython3 import gvxr
from gvxrPython3 import gvxr2json
from gvxrPython3.cil.gVXRDataReader import gVXRDataReader
from gvxrPython3.cil.JSON2gVXRDataReader import JSON2gVXRDataReader

output_path = "output_data/reading_data_from_memory"
os.makedirs(output_path, exist_ok=True)
projection_path = os.path.join(output_path, "projections")
# json_fname = os.path.join(output_path, "simulation.json")

gvxr.createOpenGLContext()
gvxr.hideWindow()

gvxr.setSourcePosition(-40.0, 0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(200, "keV", 16000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(512, 256)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Create the sample.
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        20,
        20,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Set the sample properties
for label, element, colour in [("cylinder", "Si", [1, 0, 0, 1]), ("spheres", "Fe", [0, 1, 0, 1])]:
    # gvxr.loadMeshFile(label, os.path.join(output_path, label + ".stl"), "mm")
    gvxr.setElement(label, element)
    gvxr.setColour(label, *colour)
    gvxr.addPolygonMeshAsInnerSurface(label)

gvxr.rotateNode("cylinder", 90, 1, 0, 0)
gvxr.applySOD("root", gvxr.getBestSOD("cylinder", "mm", 1.01), "mm")

number_of_projections = 300

def run_ct(path):
    gvxr.computeCTAcquisition(path,
                              "",
                              number_of_projections,
                              0, True, 360,
                              0,
                              *gvxr.getNodeAndChildrenBoundingBoxCenter("root", "mm"), "mm",
                              *gvxr.getDetectorUpVector())

def memory_workflow():
    run_ct("")                          # Projections stay in the main memory
    return gVXRDataReader().read()      # CIL reads them directly from gVXR

def disk_workflow():
    run_ct(projection_path)             # Projections are written as TIFF files
    return gVXRDataReader(projection_path).read()  # CIL reads the TIFF files back

def timeit(func, repeats=3):
    func()                              # Warm-up (shader compilation, caches, etc.)
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        data = func()
        times.append(time.perf_counter() - start)
    return data, min(times)

data_memory, time_memory = timeit(memory_workflow)
data_disk, time_disk = timeit(disk_workflow)

gvxr.destroy()

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(121); plt.title("Loaded from memory"); plt.imshow(data_memory.as_array()[0], cmap="gray"); plt.colorbar()
plt.subplot(122); plt.title("Loaded from disk");   plt.imshow(data_disk.as_array()[0], cmap="gray"); plt.colorbar()
plt.show()

In [ ]:
print(f"In memory: {time_memory:.3f} s")
print(f"From disk: {time_disk:.3f} s")
print("Same shape:", data_memory.shape == data_disk.shape)
print("Max absolute difference:", np.abs(data_memory.as_array() - data_disk.as_array()).max())

In [ ]:
from cil.processors import TransmissionAbsorptionConverter
from cil.utilities.display import show2D
# from cil.utilities.jupyter import islicer

from cil.recon import FDK # For CBCT
from cil.io import TIFFWriter

data_memory_corr = TransmissionAbsorptionConverter(white_level=data_memory.max())(data_memory)
data_disk_corr = TransmissionAbsorptionConverter(white_level=data_disk.max())(data_disk)

In [ ]:
show2D([data_memory_corr, data_disk_corr], title=["Data from RAM", "Data from Disk"])


In [ ]:
ig = data_memory_corr.geometry.get_ImageGeometry()

data_memory_FDK_reconstruction = FDK(data_memory_corr, ig).run()
data_disk_corr_FDK_reconstruction = FDK(data_disk_corr, ig).run()

In [ ]:
show2D([data_memory_FDK_reconstruction, data_disk_corr_FDK_reconstruction], title=["Data from RAM", "Data from Disk"])

---

# 11. Projection Matrices with Explicit Units

## Overview

For the 3D visualisation, projection matrix functions (i.e. `orthographic` and `perspective`) now allow explicit length units to be specified.

This reduces ambiguity when mixing datasets that use different units.

## Exercise

### Objective

Verify that equivalent lengths expressed with different units produce identical geometries.

### Steps

1. Change the visualisation window size so that it is $512 \times 256$ pixels using:
```python
gvxr.setWindowSize(512, 256)
```

2. Load a projection matrix using:

```python
gvxr.perspective(fovy, # field of view angle, in degrees, in the y-direction
    aspect_ratio, # The aspect ratio is the ratio of x (width) to y (height)
    zNear, # the distance from the viewer to the near clipping plane
    zFar, # the distance from the viewer to the far clipping plane
    aUnitOfLength # the unit of length corresponding to zNear and zFar
)
```

   ```text
   zNear = 10 mm
   zFar = 600 mm
   ```

3. Create another using:

   ```text
   zNear = 1 cm
   zFar = 60 cm
   ```

4. Create another using:

   ```text
   zNear = 0.01 m
   zFar = 0.6 m
   ```

5. Compare the resulting geometries.

### Expected output

You should observe:

- Identical viewing geometries;
- Matching field of view;
- Consistent projection behaviour regardless of chosen unit.

### Questions

- Are the geometries identical?
- Which workflow is least prone to unit-conversion errors?

### Reference Implementation

In [ ]:
from matplotlib import pyplot as plt
from gvxrPython3 import gvxr # Simulate X-ray images

gvxr.createOpenGLContext()

gvxr.setSourcePosition(-40.0,  0.0, 0.0, "cm")
gvxr.usePointSource()
gvxr.setMonoChromaticPerPixelAtSDD(0.08, "MeV", 1000)

gvxr.setDetectorPosition(10.0, 0.0, 0.0, "cm")
gvxr.setDetectorUpVector(0, 0, -1)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

# Set the sample
cylinder_radius_in_mm = 20.0
cylinder_height_in_mm = 20.0
percent_sphere_volume = 0.025
sphere_radius_in_mm = 2.0

gvxr.makeSpheresInCylinder(
        "cylinder",
        150,
        cylinder_radius_in_mm,
        cylinder_height_in_mm,
        "spheres",
        40,
        40,
        sphere_radius_in_mm,
        percent_sphere_volume,
        "mm");

# Change the colours
gvxr.setColour("cylinder", 1, 0, 0, 1)
gvxr.setColour("spheres", 0, 1, 0, 1)

# Rotate the sample. The spheres are within the cylinder, only rotate the later.
gvxr.rotateNode("cylinder", 90, 1, 0, 0);

gvxr.applySOD("cylinder", gvxr.getBestSOD("cylinder", "mm"), "mm")

gvxr.setElement("cylinder", "Si");
gvxr.setElement("spheres", "Fe");

gvxr.addPolygonMeshAsInnerSurface("cylinder");
gvxr.addPolygonMeshAsInnerSurface("spheres");

gvxr.computeXRayImage();

In [ ]:
gvxr.setWindowSize(512, 256)
gvxr.displayScene()

aspect_ratio = gvxr.getWindowSize()[0] / gvxr.getWindowSize()[1]

In [ ]:
# zNear = 10 mm
# zFar = 600 mm
gvxr.perspective(60.0, aspect_ratio, 10.0, 600.0, "mm")
gvxr.displayScene()
screenshot_mm = gvxr.takeScreenshot()

# zNear = 1 cm
# zFar = 60 cm
gvxr.perspective(60.0, aspect_ratio, 1.0, 60.0, "cm")
gvxr.displayScene()
screenshot_cm = gvxr.takeScreenshot()

# zNear = 0.01 m
# zFar = 0.6 m
gvxr.perspective(60.0, aspect_ratio, 0.01, 0.6, "m")
gvxr.displayScene()
screenshot_m = gvxr.takeScreenshot()

gvxr.destroy()

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(131)
plt.title("zNear = 10 mm, zFar = 600 mm")
plt.imshow(screenshot_mm)
plt.subplot(132)
plt.title("zNear = 1 cm, zFar = 60 cm")
plt.imshow(screenshot_cm)
plt.subplot(133)
plt.title("zNear = 0.01 cm, zFar = 0.6 m")
plt.imshow(screenshot_m)
plt.show()

---

# 12. Visualisation Improvements

## Overview

Zoom behaviour is now distance-aware and screenshots are generated in UINT8 format.

## Exercise

### Objective

Evaluate the new visualisation behaviour.

### Steps

1. Load a small object.
2. Zoom in and out repeatedly.
3. Repeat with a much larger object.
4. Capture screenshots.
5. Compare image quality and navigation responsiveness.

### Expected output

You should observe:

- Smooth and predictable zoom behaviour;
- Consistent navigation across different scene scales;
- Screenshots suitable for reports, documentation, and presentations.

### Questions

- Does zooming feel more natural?
- Is navigation consistent across differently sized datasets?
- Are the screenshots publication-ready?

### Challenge (optional)

Prepare a one-page figure showing the same object at several zoom levels and acquisition geometries using only screenshots generated by gVXR.

---

# 13. Migration Notes

The following functions have been removed:

```cpp
void setSceneRotationMatrix(const std::vector<double>& aRotationMatrix);
std::vector<double> getSceneRotationMatrix();

void setZoom(float aZoomValue);
float getZoom();
```

## Recommended Replacement

Use the new camera system instead of directly manipulating:

- Scene rotation matrices;
- Arbitrary zoom values.

The new approach is more intuitive, more robust, and better aligned with modern visualisation workflows.

---

# 14. Capstone Exercise

## Objective

Combine the new features introduced in gVXR 2.1.1 into a complete workflow.

### Steps

1. Import an XTekCT geometry.
2. Load a CAD model.
3. Explore the scene using the camera controls.
4. Adjust SOD and ODD interactively.
5. Validate positions using bounding-box queries.
6. Generate silhouettes from multiple viewpoints.
7. Capture screenshots.
8. Process data directly from memory.

### Expected output

You should obtain:

- A fully configured virtual acquisition setup;
- Interactive geometry exploration;
- A collection of silhouettes;
- Several screenshots documenting the workflow;
- A complete workflow without unnecessary intermediate files.

### Questions

- Which new feature provided the greatest productivity gain?
- Which feature is most useful for your research or teaching activities?
- How could these additions improve your existing gVXR workflows?